# Решения: Проект 20 — рынок труда IT в ЕС

Решения упражнений из `20_IT_Job_Market_EU.ipynb`. Данные (снимок 2025:
Stack Overflow Survey / Numbeo / OECD) и базовая модель воспроизведены ниже.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

SNAPSHOT = {
  'Люксембург':('LU',85000,1800,11000,0.29), 'Дания':('DK',75000,1500,6000,0.36),
  'Ирландия':('IE',70000,2100,6500,0.28), 'Германия':('DE',68000,1250,7000,0.39),
  'Нидерланды':('NL',66000,1900,7500,0.32), 'Австрия':('AT',62000,1050,8000,0.34),
  'Швеция':('SE',60000,1400,7000,0.30), 'Финляндия':('FI',58000,1150,5500,0.33),
  'Бельгия':('BE',58000,1050,4000,0.42), 'Франция':('FR',52000,1300,10000,0.29),
  'Испания':('ES',42000,1200,4500,0.24), 'Эстония':('EE',42000,800,3500,0.21),
  'Италия':('IT',40000,1200,5000,0.33), 'Польша':('PL',40000,950,4000,0.24),
  'Литва':('LT',40000,800,3000,0.29), 'Чехия':('CZ',38000,1000,5500,0.23),
  'Румыния':('RO',36000,600,2200,0.40), 'Португалия':('PT',35000,1300,5000,0.29),
  'Венгрия':('HU',32000,650,3000,0.33), 'Греция':('GR',28000,600,2800,0.30),
  'Хорватия':('HR',30000,700,3200,0.27),
}
df = pd.DataFrame([(c0, v[0], v[1], v[2], v[3], v[4]) for c0, v in SNAPSHOT.items()],
                  columns=['country','code','gross','rent','price_m2','tax_social'])

EU_MORTGAGE_RATE, DOWN, YEARS, AREA = 0.035, 0.20, 30, 60
def annual_mortgage(price_m2, rate=EU_MORTGAGE_RATE):
    P = price_m2 * AREA * (1 - DOWN); r = rate / 12; n = YEARS * 12
    return P * r * (1 + r) ** n / ((1 + r) ** n - 1) * 12

def build(df):
    df = df.copy()
    df['net_year'] = df['gross'] * (1 - df['tax_social'])
    df['home_price'] = df['price_m2'] * AREA
    df['annual_rent'] = df['rent'] * 12
    df['annual_mortgage'] = df['price_m2'].apply(annual_mortgage)
    df['dispo_rent'] = df['net_year'] - df['annual_rent']
    df['dispo_buy'] = df['net_year'] - df['annual_mortgage']
    df['price_to_income'] = df['home_price'] / df['gross']
    return df

def zscore(s): return (s - s.mean()) / s.std(ddof=0)
WEIGHTS = {'net_year':0.35,'dispo_rent':0.30,'annual_rent':-0.15,'tax_social':-0.10,'price_to_income':-0.10}
def compute_score(d, w):
    s = pd.Series(0.0, index=d.index)
    for col, wt in w.items(): s += wt * zscore(d[col])
    return s

df = build(df)
df['score'] = compute_score(df, WEIGHTS)
print('Топ-5 (базовый рейтинг):')
print(df.sort_values('score', ascending=False)[['country','net_year','dispo_rent']].head())

## Упражнение 1: Покупательная способность (PPP)

Добавим сравнительный уровень цен (Eurostat PLI, ЕС≈100) и пересчитаем реальный
располагаемый доход: `real = dispo_rent / (PLI/100)`.

In [ ]:
# Сравнительный уровень цен (Eurostat comparative price levels ~2023, EU27=100)
PLI = {'LU':149,'DK':143,'IE':141,'DE':108,'NL':116,'AT':112,'SE':121,'FI':122,
       'BE':111,'FR':111,'ES':95,'EE':89,'IT':102,'PL':60,'LT':74,'CZ':79,
       'RO':58,'PT':89,'HU':65,'GR':88,'HR':76}
d = df.copy(); d['PLI'] = d['code'].map(PLI)
d['dispo_ppp'] = d['dispo_rent'] / (d['PLI'] / 100)

comp = d.sort_values('dispo_ppp', ascending=False)[
    ['country','dispo_rent','PLI','dispo_ppp']].head(10).copy()
comp['dispo_rent'] = comp['dispo_rent'].map('€{:,.0f}'.format)
comp['dispo_ppp'] = comp['dispo_ppp'].map('{:,.0f} int$'.format)
print('Топ-10 по РЕАЛЬНОМУ располагаемому доходу (PPP):')
print(comp.to_string(index=False))
print('\nПольша, Эстония, Чехия поднимаются (низкие цены);')
print('Ирландия, Люксембург, Нидерланды опускаются (дорогая жизнь).')

## Упражнение 2: Специальные налоговые режимы

Нидерландский **30%-ruling** (30% зарплаты не облагается) и польский **IP Box** (5%
на доход от IP вместо обычной ставки). Пересчитаем NL и PL.

In [ ]:
d = df.copy()
# NL 30%-ruling: облагается только 70% дохода
nl = d[d.code == 'NL'].iloc[0]
nl_net_ruling = nl['gross'] * (0.70 * (1 - nl['tax_social']) + 0.30)
print('Нидерланды:')
print(f'  Обычно:        на руки €{nl["net_year"]:,.0f} (эфф. {nl["tax_social"]*100:.0f}%)')
print(f'  С 30%-ruling:  на руки €{nl_net_ruling:,.0f} '
      f'(эфф. {(1-nl_net_ruling/nl["gross"])*100:.0f}%)')

# PL IP Box: ставка 5% вместо обычной эффективной
pl = d[d.code == 'PL'].iloc[0]
pl_net_ipbox = pl['gross'] * (1 - 0.05)
print('\nПольша:')
print(f'  Обычно:        на руки €{pl["net_year"]:,.0f} (эфф. {pl["tax_social"]*100:.0f}%)')
print(f'  С IP Box 5%:   на руки €{pl_net_ipbox:,.0f} (эфф. 5%)')
print('\nСпец. режимы резко повышают привлекательность: NL для релокантов,')
print('PL для ИП-разработчиков на IP Box.')

## Упражнение 3: Аренда vs покупка

Рейтинг по `dispo_buy` (ипотека 3.5%) и ставка безубыточности для топ-стран.

In [ ]:
top_buy = df.sort_values('dispo_buy', ascending=False)[
    ['country','dispo_rent','annual_mortgage','dispo_buy']].head(8).copy()
for col in ['dispo_rent','annual_mortgage','dispo_buy']:
    top_buy[col] = top_buy[col].map('€{:,.0f}'.format)
print('Топ-8 по располагаемому доходу при ПОКУПКЕ (ипотека 3.5%):')
print(top_buy.to_string(index=False))

for country in ['Германия', 'Эстония']:
    row = df[df.country == country].iloc[0]
    def gap(rate): return annual_mortgage(row['price_m2'], rate) - row['annual_rent']
    lo, hi = 0.0001, 0.30
    for _ in range(60):
        mid = (lo + hi) / 2
        if gap(mid) > 0: hi = mid
        else: lo = mid
    print(f'{country}: покупка выгоднее аренды при ставке < {mid*100:.2f}%')

## Упражнение 4: Свежие данные через Eurostat API

Бесплатный API Eurostat (без ключа). Пример: House Price Index (`prc_hpi_a`).
Код защищён `try/except` и откатывается офлайн.

In [ ]:
import json, urllib.request
def fetch_eurostat_hpi():
    url = ('https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/'
           'prc_hpi_a?format=JSON&purchase=TOTAL&unit=INX_A_AVG&time=2023')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=15) as r:
        raw = json.load(r)
    geos = raw['dimension']['geo']['category']['label']
    return len(geos)

try:
    n = fetch_eurostat_hpi()
    print(f'Eurostat HPI: получены данные по {n} странам (2023).')
except Exception as e:
    print(f'Eurostat API недоступен ({type(e).__name__}); используется встроенный снимок.')
    print('При наличии интернета вернётся индекс цен на жильё по странам ЕС.')

## Упражнение 5: Свои приоритеты (налоги важнее всего)

In [ ]:
MY_WEIGHTS = {'net_year':0.25,'dispo_rent':0.20,'annual_rent':-0.10,
              'tax_social':-0.35,'price_to_income':-0.10}
d = df.copy(); d['my_score'] = compute_score(d, MY_WEIGHTS)
top5 = d.sort_values('my_score', ascending=False)[['country','tax_social','net_year','dispo_rent']].head(5).copy()
top5['tax_social'] = (top5['tax_social']*100).map('{:.0f}%'.format)
for col in ['net_year','dispo_rent']: top5[col] = top5[col].map('€{:,.0f}'.format)
print('Мой топ-5 (приоритет — низкие налоги):')
print(top5.to_string(index=False))
print('\nВперёд выходят страны с низкой нагрузкой: Эстония (21%), Чехия (23%), Испания/Польша (24%).')